In [21]:
import pandas as pd
import numpy as np
from google.cloud import storage
import os
from ast import literal_eval


In [52]:
PROJECT_ID = "hallowed-hold-454809-q7"
LOCATION = "asia-southeast1"
BUCKET_URI = "gs://namtua-movie-data"
PIPELINE_ROOT_PATH = f"{BUCKET_URI}/pipelines"
PIPELINE_NAME = "movie_pipeline.yaml"
os.environ['GOOGLE_APPLICATION_CREDENTIALS']='/Users/nam/.config/gcloud/legacy_credentials/namlearngcp@gmail.com/adc.json'

In [17]:
def download_csv(blob_path, output_path):
    blob = bucket.blob(blob_path)
    blob.download_to_filename(output_path)

# Download each file
bucket="namtua-movie-data"
metadata_path="data/recommendation_data/movies_metadata.csv"
links_small_path="data/recommendation_data/links_small.csv"
credits_path="data/recommendation_data/credits.csv"
keywords_path="data/recommendation_data/keywords.csv"
ratings_small_path="data/recommendation_data/ratings_small.csv"


client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(bucket)
download_csv(credits_path, "credits.csv")
download_csv(keywords_path, "keywords.csv")
download_csv(links_small_path, "links_small.csv")
download_csv(metadata_path, "movies_metadata.csv")
download_csv(ratings_small_path, "ratings_small.csv")

In [22]:
credits_df = pd.read_csv("credits.csv")
keywords_df = pd.read_csv("keywords.csv")
links_small_df = pd.read_csv("links_small.csv")
metadata_df = pd.read_csv("movies_metadata.csv")
ratings_small_df = pd.read_csv("ratings_small.csv")
links_small_df = links_small_df[links_small_df["tmdbId"].notnull()][
    "tmdbId"
].astype("int")

metadata_df["id"] = pd.to_numeric(
    metadata_df["id"], errors="coerce", downcast="integer"
)
metadata_df = metadata_df.dropna(subset=["id"]).astype({"id": "int"})

# chuyển cột sang định dạng số (interger), nếu dòng đó không chuyển được sang dạng số, chuyển thành NaN

# chỉ lấy tập con của metadata
metadata_df = metadata_df[metadata_df["id"].isin(links_small_df)]

keywords_df["id"] = keywords_df["id"].astype("int")
credits_df["id"] = credits_df["id"].astype("int")
metadata_df["id"] = metadata_df["id"].astype("int")

metadata_df = metadata_df.merge(credits_df, on="id")
metadata_df = metadata_df.merge(keywords_df, on="id")

metadata_small = metadata_df[
    [
        "genres",
        "id",
        "overview",
        "popularity",
        "production_companies",
        "spoken_languages",
        "title",
        "vote_average",
        "vote_count",
        "cast",
        "crew",
        "keywords",
    ]
]

numeric_cols = metadata_small.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = metadata_small.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Biến số:", numeric_cols)
print("Biến phân loại:", categorical_cols)
metadata_small["popularity"] = metadata_small["popularity"].astype("float")

features = [
    "crew",
    "cast",
    "keywords",
    "genres",
    "production_companies",
    "spoken_languages",
]
for feature in features:
    metadata_small[feature] = metadata_small[feature].apply(literal_eval)

def get_director(x):
    for i in x:
        if i["job"] == "Director":
            return i["name"]
    return np.nan

def get_list(
    x,
):  # return list feature chứa tên diễn viên, key words của bộ phim, đội ngũ làm phim, thể loại phim,...
    if isinstance(x, list):  # Kiểm tra xem x có phải là một danh sách không
        names = [i["name"] for i in x]
        if len(names) > 5:
            names = names[:5]
        return names

    return []

metadata_small["director"] = metadata_small["crew"].apply(get_director)

features = [
    "cast",
    "keywords",
    "genres",
    "production_companies",
    "spoken_languages",
]
for feature in features:
    metadata_small[feature] = metadata_small[feature].apply(get_list)

metadata_small = metadata_small.drop(columns="crew")
metadata_small["rating"] = (
    metadata_small["vote_average"] * metadata_small["popularity"]
) / metadata_small["popularity"].mean()
metadata_small["rating_class"] = pd.qcut(
    metadata_small["rating"], q=[0, 0.3, 0.6, 1.0], labels=["low", "medium", "high"]
)

ratings_small_df = ratings_small_df.drop(columns=["timestamp"])

/var/folders/nk/dqvnpjsj6r5686rdtrt_h7q40000gn/T/ipykernel_67661/4041435003.py:4: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata_df = pd.read_csv("movies_metadata.csv")
/var/folders/nk/dqvnpjsj6r5686rdtrt_h7q40000gn/T/ipykernel_67661/4041435003.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_small["popularity"] = metadata_small["popularity"].astype("float")


Biến số: ['id', 'vote_average', 'vote_count']
Biến phân loại: ['genres', 'overview', 'popularity', 'production_companies', 'spoken_languages', 'title', 'cast', 'crew', 'keywords']


/var/folders/nk/dqvnpjsj6r5686rdtrt_h7q40000gn/T/ipykernel_67661/4041435003.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_small[feature] = metadata_small[feature].apply(literal_eval)
/var/folders/nk/dqvnpjsj6r5686rdtrt_h7q40000gn/T/ipykernel_67661/4041435003.py:81: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_small["director"] = metadata_small["crew"].apply(get_director)
/var/folders/nk/dqvnpjsj6r5686rdtrt_h7q40000gn/T/ipykernel_67661/4041435003.py:91: SettingWithCopyWarning: 
A va

# Cosine Similarity

In [43]:
import spacy
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

In [40]:
df = metadata_small.copy()

# Load spaCy English model
nlp = spacy.load("en_core_web_sm")

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # Remove special characters & punctuation
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # Apply spaCy pipeline
    doc = nlp(text)

    # Lemmatize + remove stopwords + remove spaces
    tokens = [
        token.lemma_
        for token in doc
        if token.lemma_ not in ENGLISH_STOP_WORDS and not token.is_space
    ]

    return ' '.join(tokens)

def create_movie_profile(row):
    genres = ' '.join(row['genres']) if isinstance(row['genres'], list) else ''
    keywords = ' '.join(row['keywords']) if isinstance(row['keywords'], list) else ''
    overview = row['overview'] if isinstance(row['overview'], str) else ''
    cast = ' '.join(row['cast']) if isinstance(row['cast'], list) else ''
    production_companies = ' '.join(row['production_companies']) if isinstance(row['production_companies'], list) else ''
    spoken_languages = ' '.join(row['spoken_languages']) if isinstance(row['spoken_languages'], list) else ''
    director = row['director'] if isinstance(row['director'], str) else ''
    rating_class = row['rating_class'] if isinstance(row['rating_class'], str) else ''
    return f"{genres} {overview} {keywords} {cast} {production_companies} {spoken_languages} {director} {rating_class}"




In [41]:
df['movie_profile'] = df.apply(create_movie_profile, axis=1)
df['movie_profile'] = df['movie_profile'].apply(clean_text)

In [46]:
vectorizer = TfidfVectorizer(stop_words='english')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
## TODO - add code to push to GCS



['tfidf_vectorizer.pkl']

## Cosine - load and test

In [47]:
vectorizer=joblib.load('tfidf_vectorizer.pkl')
tfidf_matrix = vectorizer.fit_transform(df['movie_profile'])

# Biến đầu vào thành vector
user_query = "Today I want to watch animation about friendships"
user_query = clean_text(user_query)
user_vec = vectorizer.transform([user_query])


from sklearn.metrics.pairwise import cosine_similarity

cos_sim = cosine_similarity(user_vec, tfidf_matrix).flatten()

# Lấy top N phim
top_n = 20
top_indices = cos_sim.argsort()[-top_n:][::-1]

recommended_movies = df.iloc[top_indices][['title', 'vote_average',
                                           'genres','spoken_languages','keywords']]
recommended_movies

,title,vote_average,genres,spoken_languages,keywords
35,Across the Sea of Time,3.5,"[Adventure, History, Drama, Family]","[Pусский, English]",[]
2751,Creature Comforts,7.3,"[Animation, Comedy, Family]","[English, Deutsch]",[animation]
3981,Return to Never Land,6.1,"[Adventure, Fantasy, Animation, Family]",[English],[animation]
7794,Day & Night,7.6,"[Animation, Family]",[],"[short, pixar animation]"
7589,Death at a Funeral,5.5,[Comedy],[English],"[remake, family relationships]"
8334,"Batman: The Dark Knight Returns, Part 2",7.9,"[Action, Animation]",[English],"[dc comics, future, joker, robin, based on gra..."
8065,The Intouchables,8.2,"[Drama, Comedy]","[English, Français]","[male friendship, masseuse, friendship, aristo..."
8423,Monsters University,7.0,"[Animation, Family]",[English],"[monster, dormitory, games, animation, best fr..."
5456,Garfield,5.2,"[Animation, Comedy, Family]",[English],"[competition, moderator, lasagne, garfield]"
6546,Mrs Palfrey at The Claremont,6.8,"[Comedy, Drama]",[English],"[poetry, friendship, memory]"


# Colaborative - Build SVD Model

In [27]:
from surprise import Dataset, Reader, SVD
import joblib


In [49]:
new_user_ratings = pd.DataFrame([
    {'userId': 9999, 'movieId': 1172, 'rating': 5.0},
    {'userId': 9999, 'movieId': 1061, 'rating': 4.5},
    {'userId': 9999, 'movieId': 1029, 'rating': 4.0},
])

ratings = pd.concat([ratings_small_df, new_user_ratings], ignore_index=True)



reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

trainset = data.build_full_trainset()

# Huấn luyện SVD model
svd_model = SVD()
svd_model.fit(trainset)

joblib.dump(svd_model, 'svd_model.pkl')

## TODO - import code to push model to registry/s3


['svd_model.pkl']

In [57]:
from surprise.model_selection import cross_validate
results = cross_validate(svd_model, data, measures=["RMSE", "MAE"], cv=3, verbose=True)



Evaluating RMSE, MAE of algorithm SVD on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    0.8999  0.9036  0.9064  0.9033  0.0027  
MAE (testset)     0.6956  0.6947  0.6977  0.6960  0.0012  
Fit time          0.69    0.70    0.67    0.69    0.02    
Test time         0.22    1.35    0.12    0.56    0.56    


In [54]:
from google.cloud import storage

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket("namtua-movie-data")
gcs_path="models/svd_test.pkl"
model_path="svd_model.pkl"
blob = bucket.blob(gcs_path)

# Upload the file
blob.upload_from_filename(model_path)

## Colaborative - load and test

In [48]:
new_svd_model=joblib.load('svd_model.pkl')

# Lấy top phim từ content-based recommendation
cb_recommended_ids = df.iloc[top_indices]['id'].tolist()  # hoặc df['movieId'] nếu khớp với `ratings`

# Dự đoán rating với các phim này
svd_preds = []
for movie_id in cb_recommended_ids:
    pred = svd_model.predict(uid=164982, iid=movie_id)
    svd_preds.append((movie_id, pred.est))

# Sắp xếp theo dự đoán
svd_preds.sort(key=lambda x: x[1], reverse=True)

# Lấy top 5 phim gợi ý cuối cùng
top_5_final = [movie_id for movie_id, _ in svd_preds[:5]]
final_recommendations = df[df['id'].isin(top_5_final)][['title', 'vote_average', 'genres']]
final_recommendations

,title,vote_average,genres
35,Across the Sea of Time,3.5,"[Adventure, History, Drama, Family]"
2751,Creature Comforts,7.3,"[Animation, Comedy, Family]"
3981,Return to Never Land,6.1,"[Adventure, Fantasy, Animation, Family]"
7589,Death at a Funeral,5.5,[Comedy]
7794,Day & Night,7.6,"[Animation, Family]"


In [19]:
gcs_path="gs://namtua-movie-data/models/svd_v1.pkl"

In [24]:
bucket_name = gcs_path[5:].split('/')[0]
prefix = '/'.join(gcs_path[5:].split('/')[2:])

In [25]:
prefix

'svd_v1.pkl'